# RL Day 1 — Jupyter Notebook
**MAB → Maze1D → Maze2D**

> This notebook runs on Binder. Your session is temporary — download any results you want to keep.

Today you only need to change the parameters marked with **🔧** and observe how the agent's behavior and training curves change.

In [ ]:
# ── Setup: download RR environments and import ──────────────────
RR_ENVS_URL = 'https://raw.githubusercontent.com/colombo0718/rl-lab-group-b/master/rr_envs.py'

!rm -f rr_envs.py
!wget -q "{RR_ENVS_URL}" -O rr_envs.py
!pip install gymnasium matplotlib numpy -q

import importlib, sys, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display

if 'rr_envs' in sys.modules:
    del sys.modules['rr_envs']

from rr_envs import (
    MABEnv, Maze1DEnv, Maze2DEnv, HeliEnv, FighterEnv,
    show_env, animate_actions, animate_random, run_q_learning, plot_training, plot_maze2d_qtable, plot_maze1d_qtable,
)

def make_key_fn(env, bins=None):
    from gymnasium import spaces
    if isinstance(env.observation_space, spaces.Discrete):
        return lambda obs: int(obs)
    obs_dim = env.observation_space.shape[0]
    if bins is None:
        bins = 6
    low, high = env.observation_space.low, env.observation_space.high
    edges = [np.linspace(low[i], high[i], bins + 1) for i in range(obs_dim)]
    def to_key(obs):
        return tuple(int(np.clip(np.digitize(obs[i], edges[i]) - 1, 0, bins - 1)) for i in range(obs_dim))
    return to_key

def animate_policy(env, Q, steps=40, bins=None, delay=0.25):
    to_key = make_key_fn(env, bins=bins)
    obs, info = env.reset()
    for step in range(steps):
        clear_output(wait=True)
        display(HTML(env.render_html()))
        state = to_key(obs)
        action = max(range(env.action_space.n), key=lambda a: Q.get((state, a), 0.0))
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        print(f'step={step+1}  action={action}  reward={reward:.2f}  done={done}')
        time.sleep(delay)
        if done:
            clear_output(wait=True)
            display(HTML(env.render_html()))
            print(f'finished at step={step+1}, final reward={reward:.2f}')
            break

print('✅ RR environments ready!')


---
## Part 1 · MAB — Multi-Armed Bandit

Mirrors `MAB_en.html` on Rein Room.

**Game:** 5 slot machines (A0–A4). Each episode = 10 pulls. Pick a machine, observe its reward.

**Rewards:** `❌ = 0` · `🍬 = 1` · `🪙 = 3` · `💎 = 10`

**Modes** (`MAB_MODE`):
| Mode | What it means |
|---|---|
| `same` | All 5 machines share identical pools — no best choice |
| `fixed` | Two machines always 0, two always 1, one always 3 — best is obvious |
| `slight` | Machines have subtly different expected values — hard to detect |
| `jackpot` | One machine hides a 💎 pool — high risk, high reward |

**Key hyperparameters:**
- `EPSILON` — exploration rate: high = tries all machines randomly; low = exploits known best
- `ALPHA` — learning rate: how fast Q-values update after each pull

**T1 task:** Compare `ε = 0.9` vs `ε = 0.1`. Which converges faster? Which finds the best machine?


In [ ]:
# ═══════════════ 🔧 Change these values ═══════════════
MAB_MODE  = 'jackpot'   # 'same'=no difference  'fixed'=clear best  'slight'=subtle  'jackpot'=hidden gem
EPISODES  = 400        # number of 10-pull rounds to train
ALPHA     = 0.5        # learning rate (0–1): how fast Q-values update each pull
GAMMA     = 0.9        # discount factor: less critical here (episodes are short)
EPSILON   = 0.9        # exploration rate: 0.9=explore a lot  0.1=exploit known best
# ═══════════════════════════════════════════

env = MABEnv(mode=MAB_MODE)
obs, info = env.reset()
show_env(env)
print('Watching a random agent play MAB...')

In [ ]:
animate_random(env, steps=10, delay=0.35)

In [ ]:
env = MABEnv(mode=MAB_MODE)
rewards, lengths, Q = run_q_learning(
    env, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, n_episodes=EPISODES, bins=5, verbose=False
)
plot_training(rewards, lengths, label=f'MAB mode={MAB_MODE}, epsilon={EPSILON}', window=30)

to_key = make_key_fn(env, bins=5)
print('Learned Q-values by observed selected machine state:')
for state_action, value in sorted(Q.items()):
    print(state_action, '=>', round(value, 3))

In [ ]:
# ── T1: Compare ε = 0.9 vs ε = 0.1 ─────────────────────────────────
window = 30
fig, ax = plt.subplots(figsize=(10, 4))
smooth = lambda r: np.convolve(r, np.ones(window)/window, mode="valid")

for eps, style in [(0.9, "-"), (0.1, "--")]:
    _env = MABEnv(mode=MAB_MODE)
    _r, _, _ = run_q_learning(_env, alpha=ALPHA, gamma=GAMMA, epsilon=eps,
                               n_episodes=EPISODES, bins=5, verbose=False)
    ax.plot(smooth(_r), linestyle=style, linewidth=2, label=f"ε = {eps}")

ax.set_title(f"T1 · MAB mode={MAB_MODE} — Reward curve: ε = 0.9 vs ε = 0.1  (smoothed, window={window})")
ax.set_xlabel("Episode")
ax.set_ylabel("Total Reward")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 2 · Maze 1D — 1D Maze

Mirrors `Maze1D_en.html` on Rein Room.

**Game:** 10 cells in a row (positions 0–9). Agent starts at `START_POS`.
- Cell 0: `💣` Bomb → reward **−10**, episode ends
- Cell 9: `🏆` Goal → reward **+10**, episode ends (or **+2** in `pie` mode)

**Actions:** `0` = stay · `1` = move right · `2` = move left

**Feedback modes** (`FEEDBACK_MODE`) — small step rewards:
| Mode | Moving toward goal | Moving toward bomb |
|---|---|---|
| `none` | 0 | 0 |
| `positive` | +1 | −1 — helpful signal |
| `negative` | −1 | +1 — misleading signal, tests robustness |
| `pie` | −1 | +1 — goal gives 🍕 +2 instead of 🏆 +10 |

**T2a task:** Point out State / Action / Reward / Episode in this game.
**T2b task:** Look at the Q-Value Slice chart — pick any bar and explain why it is positive, negative, or zero.


In [ ]:
# ═══════════════ 🔧 Change these values ═══════════════
START_POS     = 4       # agent starting cell (1–8); 4 = middle of the row
FEEDBACK_MODE = 'none'  # 'none' / 'positive' / 'negative' / 'pie'  (see table above)
EPISODES      = 500     # how many episodes to train
ALPHA         = 0.5     # learning rate
GAMMA         = 0.9     # discount factor: how much future rewards matter
EPSILON       = 0.9     # exploration rate
# ═══════════════════════════════════════════

env = Maze1DEnv(start_pos=START_POS, feedback_mode=FEEDBACK_MODE)
env.reset()
show_env(env)

In [ ]:
# Watch a fixed policy: always go right
env = Maze1DEnv(start_pos=START_POS, feedback_mode=FEEDBACK_MODE)
env.reset()
animate_actions(env, [1, 1, 1, 1, 1, 1], delay=0.45)

In [ ]:
env = Maze1DEnv(start_pos=START_POS, feedback_mode=FEEDBACK_MODE)
rewards, lengths, Q = run_q_learning(
    env, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, n_episodes=EPISODES, verbose=False
)
plot_training(rewards, lengths, label=f'Maze1D feedback={FEEDBACK_MODE}', window=30)

print('Greedy policy after training:')
env = Maze1DEnv(start_pos=START_POS, feedback_mode=FEEDBACK_MODE)
animate_policy(env, Q, steps=12, delay=0.45)

In [ ]:
# ── T2b: Q-Value Slice — visualize Bellman updates ──────────────────
plot_maze1d_qtable(Q, grid_size=10)

print("Q-values per state (best action shown):")
action_name = {0: "stay", 1: "right", 2: "left"}
for s in range(10):
    vals = {a: round(Q.get((s, a), 0.0), 3) for a in range(3)}
    best_a = max(vals, key=vals.get)
    label = "💣" if s == 0 else "🏆" if s == 9 else f"pos {s}"
    print(f"  {label:6s}  best={action_name[best_a]:5s}  Q={vals}")

---
## Part 3 · Maze 2D — 2D Maze

Mirrors `Maze2D_emoji_en.html` on Rein Room.

**Game:** Grid maze. Agent starts at `(0, 0)` (bottom-left). Reach the goal for +10.

**Actions:** `0` = stay · `1` = up · `2` = down · `3` = left · `4` = right

**Levels** (`MAZE2D_LEVEL`):
| Level | Name | Grid | Goal | Notes |
|---|---|---|---|---|
| `0` | Open Field | 10×10 | (9, 9) | Default; larger state space, needs more episodes |
| `1` | Walled In | 5×5 | (4, 4) | Constrained space — path fills in faster, good for demos |

**`BINS`** sets how many buckets each axis is divided into for the Q-table.
Rule of thumb: set `BINS = grid_size` so each cell maps to exactly one bin.

**T3 task:** After training, look at the Q-table heatmap. Trace the arrow path from Start → Goal.


In [ ]:
# ═══════════════ 🔧 Change these values ═══════════════
MAZE2D_LEVEL = 0     # 0 = Open Field (10x10, goal (9,9))  |  1 = Walled In (5x5, goal (4,4))
EPISODES     = 1000  # more episodes = more complete heatmap (try 500 for level=1)
ALPHA        = 0.5   # learning rate
GAMMA        = 0.95  # high gamma = agent values reaching the distant goal
EPSILON      = 0.3   # lower epsilon = more exploitation; arrows settle into a path earlier
BINS         = 10    # match grid_size: 10 for level=0, 5 for level=1
# ═══════════════════════════════════════════

env = Maze2DEnv(level=MAZE2D_LEVEL)
env.reset()
show_env(env)

In [ ]:
# Watch a manual route: right then up
env = Maze2DEnv()
env.reset()
animate_actions(env, [4,4,4,4,1,1,1,1], delay=0.25, stop_on_done=False)

In [ ]:
env = Maze2DEnv(level=MAZE2D_LEVEL)
rewards, lengths, Q = run_q_learning(
    env, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, n_episodes=EPISODES, bins=BINS, verbose=False
)
plot_training(rewards, lengths, label=f'Maze2D level={MAZE2D_LEVEL}, epsilon={EPSILON}', window=50)
plot_maze2d_qtable(Q, bins=BINS)
